<a href="https://colab.research.google.com/github/CHAI99k/UTS_Chairil-Septa-M_14022300004_6B-BIS/blob/main/UTS_Chairil_Septa_M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'id.bmri.livin',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers torch

Now, let's load the saved reviews from the CSV file and then proceed with the sentiment analysis using the `w11wo/indonesian-roberta-base-prdect-id` model.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the reviews from the CSV file
df_reviews = pd.read_csv('ulasan_google_play.csv')

# Display the first few rows to verify
display(df_reviews.head())

In [ ]:
# Load the tokenizer and model
model_name = 'w11wo/indonesian-roberta-base-prdect-id'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Check if GPU is available and move model to GPU if it is
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Function to predict sentiment for a given text
def get_sentiment(text):
    if not isinstance(text, str): # Handle non-string inputs
        return 'NEUTRAL', 0.0

    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=1)

    # Get the predicted class (label) and its probability
    predicted_class_id = probabilities.argmax().item()
    predicted_label = model.config.id2label[predicted_class_id]
    predicted_score = probabilities[0][predicted_class_id].item()

    return predicted_label, predicted_score

# Apply sentiment analysis to each review content
df_reviews[['sentiment', 'sentiment_score']] = df_reviews['content'].apply(lambda x: pd.Series(get_sentiment(x)))

# Display the DataFrame with sentiment analysis results
display(df_reviews.head())